# KBO 가을야구 예측 전처리 실험 - dy

이 노트북은 `data/raw`와 `data/processed`에 있는 2022~2026 KBO 데이터를 기준으로 전처리, 예측, 시각화 산출물을 다시 생성하는 실행용 파일입니다.

산출물은 모두 `notebooks/experiments/dy` 폴더 안에 저장됩니다.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

EXPERIMENT_DIR = Path(r"C:\\Users\\Admin\\Documents\\GitHub\\Ml_Baseball\\notebooks\\experiments\\dy")
REPO_DIR = EXPERIMENT_DIR.parents[2]
RAW_DIR = REPO_DIR / "data" / "raw"
PROCESSED_DIR = REPO_DIR / "data" / "processed"

sys.path.insert(0, str(EXPERIMENT_DIR))

print("실험 폴더:", EXPERIMENT_DIR)
print("원본 데이터:", RAW_DIR)
print("과정 데이터:", PROCESSED_DIR)


## 1. 입력 데이터 확인

사용 데이터 범위는 2022~2026입니다. 2022~2025는 완료 시즌, 2026은 진행 중 시즌 데이터로 다룹니다.


In [ ]:
for base in [RAW_DIR, PROCESSED_DIR]:
    print(f"\n== {base} ==")
    for year in range(2022, 2027):
        files = sorted((base / str(year)).glob("*.csv"))
        print(year, len(files), "files")


## 2. 전체 파이프라인 실행

아래 셀은 raw/processed 데이터를 읽어서 모델 데이터셋, 2026 예측 결과, 검증 결과, 발표용 PNG, HTML 대시보드를 한 번에 생성합니다.


In [ ]:
import dy_experiment_pipeline as pipe

summary = pipe.run_all()
summary


## 3. 모델 학습용 데이터 확인

`kbo_outputs/model_dataset_2022_2026.csv`는 예측에 사용되는 최종 학습/예측용 스냅샷 데이터입니다.


In [ ]:
model_df = pd.read_csv(EXPERIMENT_DIR / "kbo_outputs" / "model_dataset_2022_2026.csv", encoding="utf-8-sig")
print(model_df.shape)
model_df[["season", "date", "team", "rank", "win_rate", "postseason", "source"]].head()


## 4. 2026 가을야구 예측 결과

현재 진행 중인 2026 시즌 기준 팀별 가을야구 진출 확률입니다.


In [ ]:
pred = pd.read_csv(EXPERIMENT_DIR / "kbo_outputs" / "2026_postseason_predictions.csv", encoding="utf-8-sig")
pred[["team", "rank", "wins", "losses", "win_rate", "postseason_probability_pct", "prediction_label"]]


## 5. 검증 및 4월 인사이트

완료 시즌 기준 검증 결과와 4월 순위가 최종 순위와 얼마나 맞았는지 확인합니다.


In [ ]:
validation = pd.read_csv(EXPERIMENT_DIR / "kbo_outputs" / "validation_leave_one_season.csv", encoding="utf-8-sig")
april = pd.read_csv(EXPERIMENT_DIR / "kbo_outputs" / "april_rank_insight.csv", encoding="utf-8-sig")
display(validation)
display(april)


## 6. 최종 산출물 위치

아래 파일들이 발표/제출용 결과입니다.

- `2026_가을야구_예측결과.csv`
- `team_master_2022_2026.csv`
- `리그_타격환경.csv`, `리그_투구환경.csv`
- `01_*.png` ~ `09_*.png`
- `kbo_2022_2026_master_dashboard.html`


In [ ]:
for p in sorted(EXPERIMENT_DIR.glob("*.csv")):
    print(p.name, p.stat().st_size)

print("\nPNG files")
for p in sorted(EXPERIMENT_DIR.glob("0*.png")):
    print(p.name, p.stat().st_size)

html_path = EXPERIMENT_DIR / "kbo_2022_2026_master_dashboard.html"
print("\nHTML:", "file:///" + str(html_path).replace("\\", "/"))
